# Prepare - immutable training and metrics

**Do not edit during Auto-R&D.** This notebook validates materialized hierarchical datasets and defines the fixed topology, training loop, permutation-invariant correspondence metrics, and hierarchy/topology metrics. Dataset loading and generation belong in a provider notebook such as `Dataset.ipynb`.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from collections.abc import Mapping
import numpy as np
from scipy.optimize import linear_sum_assignment

EPS = 1e-12

In [ ]:
@dataclass
class HierarchyDataset:
    name: str
    x: np.ndarray
    labels_by_level: tuple[np.ndarray, ...]
    k_levels: tuple[int, ...]
    parent_maps: tuple[np.ndarray, ...]
    metadata: dict = field(default_factory=dict)


def prepare_dataset(raw) -> HierarchyDataset:
    if isinstance(raw, HierarchyDataset):
        return raw
    if not isinstance(raw, Mapping):
        raise TypeError('dataset provider must return a mapping or HierarchyDataset')

    required = {'name', 'x', 'labels_by_level', 'k_levels', 'parent_maps'}
    missing = sorted(required.difference(raw))
    if missing:
        raise ValueError(f'dataset is missing required fields: {missing}')

    x = np.asarray(raw['x'], dtype=float)
    if x.ndim != 2 or len(x) == 0 or x.shape[1] == 0:
        raise ValueError('x must have shape [nonzero samples, nonzero features]')
    if not np.all(np.isfinite(x)):
        raise ValueError('x contains NaN or Inf')

    k_levels = tuple(int(k) for k in raw['k_levels'])
    labels = tuple(np.asarray(z) for z in raw['labels_by_level'])
    if not k_levels or len(labels) != len(k_levels):
        raise ValueError('labels_by_level and k_levels must have equal nonzero length')
    if any(k <= 0 for k in k_levels):
        raise ValueError('all level cardinalities must be positive')

    checked_labels = []
    for level, (label, cardinality) in enumerate(zip(labels, k_levels)):
        if label.ndim != 1 or len(label) != len(x):
            raise ValueError(f'labels for level {level} must have one entry per sample')
        if not np.issubdtype(label.dtype, np.integer):
            raise TypeError(f'labels for level {level} must be integers')
        label = label.astype(int, copy=False)
        if np.any(label < 0) or np.any(label >= cardinality):
            raise ValueError(f'labels for level {level} are outside [0, {cardinality})')
        checked_labels.append(label)

    parent_maps = tuple(np.asarray(p) for p in raw['parent_maps'])
    if len(parent_maps) != len(k_levels) - 1:
        raise ValueError('one parent map is required for each adjacent level pair')
    checked_parent_maps = []
    for level, parent_map in enumerate(parent_maps):
        if parent_map.ndim != 1 or len(parent_map) != k_levels[level]:
            raise ValueError(f'parent map {level} must have one entry per child node')
        if not np.issubdtype(parent_map.dtype, np.integer):
            raise TypeError(f'parent map {level} must contain integers')
        parent_map = parent_map.astype(int, copy=False)
        if np.any(parent_map < 0) or np.any(parent_map >= k_levels[level + 1]):
            raise ValueError(f'parent map {level} refers to an invalid parent node')
        checked_parent_maps.append(parent_map)

    metadata = dict(raw.get('metadata', {}))
    return HierarchyDataset(
        name=str(raw['name']),
        x=x,
        labels_by_level=tuple(checked_labels),
        k_levels=k_levels,
        parent_maps=tuple(checked_parent_maps),
        metadata=metadata,
    )

In [ ]:
def correspondence(activity, labels, k):
    """Row-normalized conditional-mean selectivity C[neuron, latent_node]."""
    n = activity.shape[1]
    C = np.zeros((n, k), dtype=float)
    for j in range(k):
        mask = labels == j
        if np.any(mask):
            C[:, j] = activity[mask].mean(axis=0)
    C = C / (C.max(axis=1, keepdims=True) + EPS)
    return C


def optimal_permutation_metrics(C):
    """Permutation-invariant assignment for square or rectangular C."""
    rows, cols = linear_sum_assignment(-C)
    matched = C[rows, cols]
    diag = float(np.mean(matched)) if len(matched) else 0.0
    mask = np.ones_like(C, dtype=bool)
    mask[rows, cols] = False
    off = float(np.mean(np.abs(C[mask]))) if np.any(mask) else 0.0
    dominance = diag - off
    coverage = len(matched) / max(C.shape)
    return dict(
        rows=rows, cols=cols, diag=diag, offdiag=off,
        dominance=dominance, coverage=coverage, score=coverage * dominance,
    )


def aligned_square_matrix(C):
    """Return a square optimally aligned view when C itself is square."""
    assert C.shape[0] == C.shape[1]
    match = optimal_permutation_metrics(C)
    aligned = np.zeros_like(C)
    for row, col in zip(match['rows'], match['cols']):
        aligned[col] = C[row]
    return aligned, match

In [ ]:
def parent_incidence(parent_map, n_parent):
    adjacency = np.zeros((n_parent, len(parent_map)), dtype=float)
    adjacency[parent_map, np.arange(len(parent_map))] = 1.0
    return adjacency


def neural_parent_relation(child_act, parent_act):
    """Empirical parent x child association from co-activity."""
    relation = (parent_act.T @ child_act) / max(1, len(child_act))
    return relation / (relation.max(axis=0, keepdims=True) + EPS)


def topology_score(true_A, learned_A, child_match, parent_match):
    """Compare adjacency after independently aligning adjacent layers."""
    child_map = {int(c): int(r) for r, c in zip(child_match['rows'], child_match['cols'])}
    parent_map = {int(c): int(r) for r, c in zip(parent_match['rows'], parent_match['cols'])}
    if len(child_map) < true_A.shape[1] or len(parent_map) < true_A.shape[0]:
        return 0.0
    aligned = np.zeros_like(true_A)
    for parent in range(true_A.shape[0]):
        for child in range(true_A.shape[1]):
            aligned[parent, child] = learned_A[parent_map[parent], child_map[child]]
    positive = aligned[true_A > 0].mean() if np.any(true_A > 0) else 0.0
    negative = aligned[true_A == 0].mean() if np.any(true_A == 0) else 0.0
    return float(positive - negative)

In [ ]:
def normalize_input(x):
    centered = x - x.mean(axis=0, keepdims=True)
    return centered / (centered.std(axis=0, keepdims=True) + EPS)


def fixed_topology(dataset):
    return tuple(dataset.k_levels)


def train_rule_on_dataset(
    rule_module, raw_dataset, training_seed=0, epochs=12, batch_size=128, cfg=None
):
    dataset = prepare_dataset(raw_dataset)
    rng = np.random.default_rng(training_seed + 10000)
    x = normalize_input(dataset.x)
    layer_sizes = fixed_topology(dataset)
    weights = []
    states = []
    previous_dim = x.shape[1]
    for neuron_count in layer_sizes:
        weights.append(
            rng.normal(0, 1 / np.sqrt(previous_dim), size=(neuron_count, previous_dim))
        )
        states.append(rule_module.init_state(previous_dim, neuron_count, rng))
        previous_dim = neuron_count
    if cfg is None:
        cfg = rule_module.RuleConfig()

    for _ in range(epochs):
        order = rng.permutation(len(x))
        for start in range(0, len(x), batch_size):
            activity = x[order[start:start + batch_size]]
            for layer in range(len(weights)):
                output = rule_module.activate(activity, weights[layer], states[layer], cfg)
                weights[layer], states[layer] = rule_module.update(
                    activity, output, weights[layer], states[layer], cfg
                )
                if not np.all(np.isfinite(weights[layer])):
                    raise FloatingPointError('non-finite weights')
                norms = np.linalg.norm(weights[layer], axis=1, keepdims=True) + EPS
                weights[layer] = weights[layer] / np.maximum(1.0, norms / 5.0)
                activity = output

    activities = []
    activity = x
    for layer in range(len(weights)):
        activity = rule_module.activate(activity, weights[layer], states[layer], cfg)
        activities.append(activity)
    return dataset, activities, weights, states


def evaluate_dataset(rule_module, raw_dataset, training_seed=0, cfg=None):
    dataset, activities, weights, states = train_rule_on_dataset(
        rule_module, raw_dataset, training_seed=training_seed, cfg=cfg
    )
    level_count = len(activities)
    level_scores = []
    matches = []
    cross = np.zeros((level_count, level_count))
    for neural_level in range(level_count):
        for generative_level in range(level_count):
            C = correspondence(
                activities[neural_level],
                dataset.labels_by_level[generative_level],
                dataset.k_levels[generative_level],
            )
            cross[neural_level, generative_level] = optimal_permutation_metrics(C)['score']
        C = correspondence(
            activities[neural_level],
            dataset.labels_by_level[neural_level],
            dataset.k_levels[neural_level],
        )
        match = optimal_permutation_metrics(C)
        matches.append(match)
        level_scores.append(match['score'])

    specificity = float(np.mean([
        cross[level, level] - max(
            [cross[level, other] for other in range(level_count) if other != level],
            default=0,
        )
        for level in range(level_count)
    ]))
    topology_scores = []
    for level in range(level_count - 1):
        true_adjacency = parent_incidence(
            dataset.parent_maps[level], dataset.k_levels[level + 1]
        )
        learned_adjacency = neural_parent_relation(
            activities[level], activities[level + 1]
        )
        topology_scores.append(topology_score(
            true_adjacency, learned_adjacency, matches[level], matches[level + 1]
        ))

    activity_means = np.concatenate([a.mean(axis=0) for a in activities])
    stable = float(all(
        np.all(np.isfinite(w)) and np.max(np.linalg.norm(w, axis=1)) <= 5.0001
        for w in weights
    ))
    hierarchy = float(np.mean(level_scores))
    topology = float(np.mean(topology_scores)) if topology_scores else 0.0
    overall = 0.55 * hierarchy + 0.20 * specificity + 0.25 * topology
    return {
        'overall': overall,
        'hierarchy': hierarchy,
        'specificity': specificity,
        'topology': topology,
        'cross': cross,
        'level_scores': level_scores,
        'stable': stable,
        'activity_mean': float(activity_means.mean()),
    }

## Scientific interpretation

The fixed topology uses the cardinalities declared by the selected dataset provider. The default provider gives the network the correct number of neurons at every level, which isolates the learning-rule problem. Later providers can expose over-provisioned or otherwise altered cardinalities, but such a change defines a different benchmark and must be evaluated separately.